In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

In [2]:
spark = SparkSession.builder \
    .appName("CryptoRealTimeETL-Osoba3") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [3]:
crypto_schema = StructType([
    StructField("timestamp", LongType(), False),
    StructField("symbol", StringType(), False),
    StructField("price", DoubleType(), False)
])

In [9]:
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "crypto-stream") \
    .option("startingOffsets", "latest") \
    .load()

In [10]:
parsed_stream = raw_stream \
    .selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json(col("json_payload"), crypto_schema).alias("data")) \
    .select("data.*") \
    .withColumn("timestamp", from_unixtime(col("timestamp")).cast(TimestampType())) \
    .filter(
        col("price").isNotNull() & 
        (col("price") > 0) & 
        col("symbol").isNotNull() & 
        (col("symbol") != "") &
        col("timestamp").isNotNull()
    )

In [15]:
query = parsed_stream.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("clean_crypto_stream") \
    .start()

print("Potok PySpark na porcie działa prawidłowo i nasłuchuje Kafki!")

Potok PySpark na porcie działa prawidłowo i nasłuchuje Kafki!


In [13]:
spark.sql("SELECT * FROM clean_crypto_stream ORDER BY timestamp DESC LIMIT 10").show(truncate=False)

+---------+------+-----+
|timestamp|symbol|price|
+---------+------+-----+
+---------+------+-----+



In [17]:
!git add consumer.ipynb
!git commit -m "consumer"
!git push origin main

fatal: not a git repository (or any parent up to mount point /home/jovyan)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /home/jovyan)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /home/jovyan)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [18]:
!cd ~/Desktop/projekt-kafka-RTA

/bin/bash: line 1: cd: /home/jovyan/Desktop/projekt-kafka-RTA: No such file or directory
